# ***Setup***

In [ ]:
# Cell 1 — نصب وابستگی‌ها (اجرا در Colab)
!pip install --upgrade pip setuptools wheel

# هستهٔ LangChain و افزونه‌های community (برای loader/embeddings/vectorstores)
!pip install -U langchain langchain-core langchain-community

# ادغام‌های مورد نیاز: Tavily و Together (LangChain integrations)
!pip install -U langchain-tavily langchain-together "langchain[tavily]"

# ابزارهای مدل/امبدینگ و کمکی
!pip install -U transformers huggingface-hub
!pip install -U faiss-cpu
!pip install -U tiktoken pydantic
!pip install -U beautifulsoup4 requests tqdm pypdf

# (اختیاری) اگر می‌خواهی LangGraph هم نصب شود:
!pip install -U langgraph


In [ ]:
import os, re, io, json, time, requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from pathlib import Path

# پیش‌فرض‌ها رو None می‌ذاریم و تلاش می‌کنیم چند مسیر مختلف رو امتحان کنیم
PyPDFLoader = None
UnstructuredURLLoader = None
RecursiveCharacterTextSplitter = None
HuggingFaceEmbeddings = None
FAISS = None
Document = None
TavilySearch = None
ChatTogether = None
LLMChain = None
ChatPromptTemplate = None
SystemMessagePromptTemplate = None
HumanMessagePromptTemplate = None
BaseChatModel = None
faiss = None

_import_log = {}

# 1) PyPDFLoader / UnstructuredURLLoader
try:
    from langchain_community.document_loaders import PyPDFLoader, UnstructuredURLLoader
    _import_log['loader'] = "langchain_community.document_loaders"
except Exception as e1:
    try:
        from langchain.document_loaders import PyPDFLoader, UnstructuredURLLoader
        _import_log['loader'] = "langchain.document_loaders"
    except Exception as e2:
        PyPDFLoader = None
        UnstructuredURLLoader = None
        _import_log['loader'] = f"FAILED: {e1}; {e2}"

# 2) Text splitter (should be in langchain)
try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    _import_log['text_splitter'] = "langchain.text_splitter"
except Exception as e:
    RecursiveCharacterTextSplitter = None
    _import_log['text_splitter'] = f"FAILED: {e}"

# 3) HuggingFaceEmbeddings
try:
    from langchain_community.embeddings import HuggingFaceEmbeddings
    _import_log['embeddings'] = "langchain_community.embeddings"
except Exception:
    try:
        from langchain.embeddings import HuggingFaceEmbeddings
        _import_log['embeddings'] = "langchain.embeddings"
    except Exception as e:
        HuggingFaceEmbeddings = None
        _import_log['embeddings'] = f"FAILED: {e}"

# 4) FAISS vectorstore
try:
    from langchain_community.vectorstores import FAISS
    _import_log['vectorstore'] = "langchain_community.vectorstores"
except Exception:
    try:
        from langchain.vectorstores import FAISS
        _import_log['vectorstore'] = "langchain.vectorstores"
    except Exception as e:
        FAISS = None
        _import_log['vectorstore'] = f"FAILED: {e}"

# 5) Document schema
try:
    from langchain.schema import Document
    _import_log['schema.Document'] = "langchain.schema"
except Exception as e:
    Document = None
    _import_log['schema.Document'] = f"FAILED: {e}"

# 6) TavilySearch
try:
    from langchain_tavily import TavilySearch
    _import_log['tavily'] = "langchain_tavily"
except Exception:
    try:
        from langchain_tavily.tavily_search import TavilySearch
        _import_log['tavily'] = "langchain_tavily.tavily_search"
    except Exception as e:
        TavilySearch = None
        _import_log['tavily'] = f"FAILED: {e}"

# 7) ChatTogether (Together/OpenRouter integration)
try:
    from langchain_together import ChatTogether
    _import_log['together'] = "langchain_together"
except Exception:
    try:
        from langchain_together.chat_models import ChatTogether
        _import_log['together'] = "langchain_together.chat_models"
    except Exception as e:
        ChatTogether = None
        _import_log['together'] = f"FAILED: {e}"

# 8) LLMChain
try:
    from langchain.chains import LLMChain
    _import_log['LLMChain'] = "langchain.chains"
except Exception:
    try:
        from langchain import LLMChain
        _import_log['LLMChain'] = "langchain"
    except Exception as e:
        LLMChain = None
        _import_log['LLMChain'] = f"FAILED: {e}"

# 9) ChatPromptTemplate and message templates
try:
    from langchain.prompts.chat import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
    _import_log['prompts.chat'] = "langchain.prompts.chat"
except Exception:
    try:
        from langchain.prompts import ChatPromptTemplate
        from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
        _import_log['prompts.chat'] = "langchain.prompts"
    except Exception as e:
        ChatPromptTemplate = SystemMessagePromptTemplate = HumanMessagePromptTemplate = None
        _import_log['prompts.chat'] = f"FAILED: {e}"

# 10) BaseChatModel (optional)
try:
    from langchain_core.language_models.chat_models import BaseChatModel
    _import_log['BaseChatModel'] = "langchain_core.language_models.chat_models"
except Exception:
    try:
        from langchain.chat_models.base import BaseChatModel
        _import_log['BaseChatModel'] = "langchain.chat_models.base"
    except Exception as e:
        BaseChatModel = None
        _import_log['BaseChatModel'] = f"FAILED: {e}"

# 11) faiss Python binding
try:
    import faiss
    _import_log['faiss'] = "import faiss"
except Exception as e:
    faiss = None
    _import_log['faiss'] = f"FAILED: {e}"

# 12) pydantic (should be present)
try:
    from pydantic import BaseModel, validator
    _import_log['pydantic'] = "ok"
except Exception as e:
    _import_log['pydantic'] = f"FAILED: {e}"

# ایجاد دایرکتوری‌های مورد نیاز
BASE_DIR = Path("slp3_data")
BASE_DIR.mkdir(exist_ok=True)
PDF_DIR = BASE_DIR / "pdfs"
PDF_DIR.mkdir(exist_ok=True)
CHUNKS_DIR = BASE_DIR / "chunks"
CHUNKS_DIR.mkdir(exist_ok=True)
VECTOR_DIR = BASE_DIR / "faiss_index"
VECTOR_DIR.mkdir(exist_ok=True)

# نمای کلی از وضعیت ایمپورت‌ها (برای دیباگ)
print("=== Import status summary ===")
for k, v in _import_log.items():
    print(f"{k:20}: {v}")
print("="*30)

# اگر هرکدوم None هست و نیاز مبرم داری، تو کنسول بفرست تا دقیق رفع کنم.
missing = [k for k,v in _import_log.items() if str(v).startswith("FAILED")]
if missing:
    print("\nWarning: Some imports failed. Missing/failed items:")
    for m in missing:
        print(" -", m)
    print("\nIf you see failures, try re-running the install cell, سپس دوباره این cell را اجرا کن.")
    print("اگر بعد از نصب هنوز خطا دیدی، Runtime -> Restart runtime در Colab انجام بده و دوباره این cell را اجرا کن.")
else:
    print("\nAll critical imports look OK. آماده‌ایم ادامه بدیم :)")


# ***API***

In [ ]:
import os
from getpass import getpass

if not os.environ.get("TAVILY_API_KEY"):
    print("Enter your Tavily API key (input is hidden):")
    os.environ["TAVILY_API_KEY"] = getpass("TAVILY_API_KEY:\n")

if not os.environ.get("OPENROUTER_API_KEY"):
    print("Enter your OpenRouter API key (input is hidden):")
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY:\n")


# ***imports***

In [ ]:
import os
import re
import io
import json
import time
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from pathlib import Path

# LangChain community loaders & tools
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document

# Tavily integration (langchain-tavily)
from langchain_tavily import TavilySearch

# TogetherAI integration
from langchain_together import ChatTogether

# For chains and prompts
from langchain.chains import LLMChain
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.prompts.chat import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)

# BaseChatModel
from langchain_core.language_models.chat_models import BaseChatModel

# Pydantic for parser
from pydantic import BaseModel, validator

# FAISS utilities
import faiss

# utility dirs
BASE_DIR = Path("slp3_data")
BASE_DIR.mkdir(exist_ok=True)
PDF_DIR = BASE_DIR / "pdfs"
PDF_DIR.mkdir(exist_ok=True)
CHUNKS_DIR = BASE_DIR / "chunks"
CHUNKS_DIR.mkdir(exist_ok=True)
VECTOR_DIR = BASE_DIR / "faiss_index"
VECTOR_DIR.mkdir(exist_ok=True)


# ***scrape chapter links from the SLP3 page and download PDFs***

In [ ]:
BASE_URL = "https://stanford.edu/~jurafsky/slp3/"

resp = requests.get(BASE_URL)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")
links = []
for a in soup.find_all("a", href=True):
    href = a["href"]
    # معمولاً فصل‌ها به صورت pdf یا .pdf در لینک‌ها هستند یا نام chapter در متن
    if href.lower().endswith(".pdf") or "chapter" in href.lower() or "ch" in href.lower():
        full = href if href.startswith("http") else requests.compat.urljoin(BASE_URL, href)
        links.append(full)

# پاکسازی و یکتا کردن
links = list(dict.fromkeys(links))
print(f"Found {len(links)} possible links (some may not be chapter PDFs).")
for i, l in enumerate(links[:30], 1):
    print(i, l)

# دانلود فایل‌های pdf مرتبط (فیلتر ساده: فقط فایل‌های pdf)
pdf_links = [l for l in links if l.lower().endswith(".pdf")]
print(f"\nDownloading {len(pdf_links)} pdf files...")
for url in tqdm(pdf_links):
    fn = PDF_DIR / Path(url).name
    if fn.exists():
        continue
    r = requests.get(url)
    if r.status_code == 200 and b"%PDF" in r.content[:1024]:
        fn.write_bytes(r.content)
    else:
        print("Skipped (not a pdf?):", url)
print("Download complete. PDFs saved to:", PDF_DIR)


# ***load PDFs with PyPDFLoader and split into chunks***

In [ ]:
from langchain.document_loaders import PyPDFLoader

all_docs = []
for pdf_file in PDF_DIR.glob("*.pdf"):
    try:
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()  # returns list of langchain Document objects (page-level)
        # Add metadata to each doc for provenance
        for d in docs:
            d.metadata = dict(d.metadata or {})
            d.metadata.update({"source_pdf": pdf_file.name})
        all_docs.extend(docs)
        print(f"Loaded {len(docs)} pages from {pdf_file.name}")
    except Exception as e:
        print("Failed to load", pdf_file, e)

print(f"Total loaded docs (pages): {len(all_docs)}")

# Text splitting parameters (پیشنهادی شما: size_chunk=1024, overlap_chunk=64)
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 64

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""]
)

# RecursiveCharacterTextSplitter متن را به صورت بازگشتی برش می‌دهد: ابتدا سعی می‌کند روی
# جداکننده‌های با اولویت بالاتر (مثل فقره‌ها/پاراگراف‌ها) برش بزند و اگر قطعه بزرگ‌تر
# از chunk_size باقی بماند، جداکنندهٔ بعدی را امتحان می‌کند و در نهایت متن را به اندازه‌های
# مناسب تقسیم می‌کند. دلیل این کار: حفظ پیوستگی معنا در هر بخش و جلوگیری از بریدن
# جملات/فقرات می‌باشد. اگر تکه‌ها خیلی بزرگ باشند، مدل در درک دقیق متن دچار مشکل شده و
# قیمت محاسباتی افزایش می‌یابد؛ اگر خیلی کوچک باشند، زمینهٔ (context) از بین می‌رود
# و بازیابی مرتبط دشوار می‌شود.

# تبدیل docs صفحه‌ای به chunks
chunked_documents = []
for d in tqdm(all_docs, desc="Chunking pages"):
    text = d.page_content
    splits = text_splitter.split_text(text)
    for i, s in enumerate(splits):
        meta = dict(d.metadata or {})
        meta.update({"chunk_index": i})
        chunked_documents.append(Document(page_content=s, metadata=meta))

print("Total chunks produced:", len(chunked_documents))

# ذخیرهٔ مختصر از chunks روی دیسک به صورت JSONL برای بازبینی (اختیاری)
chunks_jsonl = CHUNKS_DIR / "chunks.jsonl"
with open(chunks_jsonl, "w", encoding="utf-8") as f:
    for doc in chunked_documents:
        f.write(json.dumps({"text": doc.page_content, "metadata": doc.metadata}, ensure_ascii=False) + "\n")
print("Wrote chunks to", chunks_jsonl)


# ***embeddings (HuggingFaceEmbeddings)***

In [ ]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # نمونهٔ مناسب برای سرعت و دقت

print("Using embedding model:", EMBEDDING_MODEL_NAME)
hf_emb = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

# تبدیل متون به بردارها و ساخت FAISS index
texts = [d.page_content for d in chunked_documents]
metadatas = [d.metadata for d in chunked_documents]

print("Creating FAISS index (this may take a while)...")
vectorstore = FAISS.from_texts(texts, hf_emb, metadatas=metadatas)

# ذخیرهٔ ایندکس روی دیسک
faiss_index_path = VECTOR_DIR / "faiss_index"
vectorstore.save_local(str(faiss_index_path))
print("FAISS index saved to", faiss_index_path)


# ***Loading the index and creating the retriever***

In [ ]:
from langchain_community.vectorstores import FAISS

faiss_dir = str(faiss_index_path)

# اینجا allow_dangerous_deserialization اضافه شده
vectorstore_loaded = FAISS.load_local(
    faiss_dir,
    hf_emb,
    allow_dangerous_deserialization=True
)

retriever = vectorstore_loaded.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# تست بازیاب با سه پرسش
queries = [
    "What are the main components of the hidden Markov model described in the textbook?",
    "Explain how a binary search tree works and its average-case complexity.",
    "Who is the president of Bolivia?"
]

print("\nRetriever test results:\n")
for q in queries:
    docs = retriever.get_relevant_documents(q)
    print("Query:", q)
    if not docs:
        print("  No docs returned.")
    else:
        for i, doc in enumerate(docs[:3], 1):
            src = doc.metadata.get("source_pdf", "unknown")
            preview = doc.page_content[:300].replace("\n", " ")
            print(f"  Doc {i} (source={src}): {preview}...\n")
    print("-" * 60)



# ***Definition of ChatOpenAI (LLM model from OpenRouter) with temperature=0***

In [ ]:
import os
from langchain.chat_models import ChatOpenAI

# کلیدت رو اینجا بذار
os.environ["OPENROUTER_API_KEY"] = " "

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct:free"

# استفاده از ChatOpenAI با تنظیمات OpenRouter
chat_llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0.0,
    max_tokens=512,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1"
)

print("ChatOpenAI LLM created - Model name:", MODEL_NAME)




In [ ]:
import json
from typing import Literal
from pydantic import BaseModel, Field
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.output_parsers import PydanticOutputParser
from langchain.chains import LLMChain

# 1) تعریف مدل پایتانیک
class RouterOutput(BaseModel):
    tool: Literal["VectorStore", "SearchEngine", "None"] = Field(
        description="The tool chosen by LLM for routing the question"
    )

# 2) ساخت parser
router_parser = PydanticOutputParser(pydantic_object=RouterOutput)

# 3) تعریف prompt با دستورالعمل خروجی
router_system = """You are a classification assistant.
Given a user query, decide whether the best tool to answer is:
- "VectorStore": for questions about the Stanford textbook content
- "SearchEngine": for Computer science related topics but outside the field of Stanford textbook content witch is nlp
- "None": for General questions

{format_instructions}
"""

router_user = """User query: {user_query}"""

chat_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(router_system),
    HumanMessagePromptTemplate.from_template(router_user)
]).partial(format_instructions=router_parser.get_format_instructions())

# 4) ساخت Chain
router_chain = LLMChain(
    llm=chat_llm,
    prompt=chat_prompt,
    output_parser=router_parser,
    output_key="router_output"
)

# 5) تابع wrapper برای اجرای router
def run_router(query: str):
    return router_chain.run({"user_query": query})

# --- تست ---
for q in [
    "what is transformer",
    "How to implement a binary search tree?",
    "What's the capital of France?"

]:
    try:
        out = run_router(q)
        print("Q:", q)
        print(" Router decision:", out.tool)
    except Exception as e:
        print(" Router parse error:", e)
    print("-" * 40)


# ***Tavily search tool chain (chain_engine_search)***

In [ ]:
#
!pip install -U langchain-tavily langchain-together "langchain[tavily]"

# 2) ایمپورت‌ها و تنظیم کلید (امن‌تر: با getpass)
import os
import getpass
from typing import List, Dict, Any

try:
    from langchain_tavily import TavilySearch
except Exception as e:
    raise ImportError("خطا در وارد کردن TavilySearch. مطمئن شو که پکیج 'langchain-tavily' نصب شده است.") from e

try:
    from langchain.schema import Document
except Exception as e:
    raise ImportError("خطا در وارد کردن Document از langchain.schema.") from e

# ===== تنظیم API Key =====
# اگر قبلاً ست نشده، از کاربر کلید را بپرس (امن‌تر از هاردکد)
if "TAVILY_API_KEY" not in os.environ or not os.environ["TAVILY_API_KEY"] or os.environ["TAVILY_API_KEY"] == "YOUR_TAVILY_API_KEY_HERE":
    # این روش امن‌تر است؛ کلید در لاگ ذخیره نمیشود
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your TAVILY_API_KEY: ").strip()

if not os.environ.get("TAVILY_API_KEY"):
    raise RuntimeError("TAVILY_API_KEY تنظیم نشده — لطفاً کلید را وارد کن.")

# ===== ساخت TavilySearch instance =====
tavily = TavilySearch(
    api_key=os.environ["TAVILY_API_KEY"],
    max_results=5,
    include_raw_content="markdown"
)

# ===== تابع کمکی برای تبدیل خروجی Tavily به Documents =====
def tavily_results_to_documents(tavily_response: Any) -> List[Document]:
    """
    tavily_response ممکنه یک dict با کلید 'results'، یا مستقیم یک لیست از آیتم‌ها باشه.
    هر آیتم ممکنه فیلدهای متفاوتی داشته باشه (content, snippet, summary, text).
    """
    docs: List[Document] = []
    results = None

    if isinstance(tavily_response, dict):
        # معمولاً {'results': [...]}
        results = tavily_response.get("results") or tavily_response.get("data") or []
    elif isinstance(tavily_response, list):
        results = tavily_response
    else:
        # پیش‌بینی حالت‌های دیگر
        results = []

    for r in results[:10]:
        # استخراج محتوای قابل نمایش با ترتیب اولویت
        content = (
            r.get("content")
            if isinstance(r, dict) and r.get("content")
            else r.get("snippet")
            if isinstance(r, dict) and r.get("snippet")
            else r.get("summary")
            if isinstance(r, dict) and r.get("summary")
            else r.get("text")
            if isinstance(r, dict) and r.get("text")
            else str(r)
        )
        metadata = {}
        if isinstance(r, dict):
            metadata["source_url"] = r.get("url") or r.get("link") or r.get("source")
            metadata["title"] = r.get("title")
        docs.append(Document(page_content=content or "", metadata={k:v for k,v in metadata.items() if v}))
    return docs

# ===== کلاس Retriever واقعی برای Tavily با هندل خطا =====
class TavilyRetriever:
    def __init__(self, tavily_instance: TavilySearch):
        self.tavily = tavily_instance

    def get_relevant_documents(self, query: str) -> List[Document]:
        try:
            resp = self.tavily.run(query)
        except Exception as e:
            # خطا در فراخوانی API — پیام واضح به کاربر
            raise RuntimeError(f"خطا هنگام فراخوانی Tavily: {e}") from e

        # تبدیل خروجی Tavily به Documents
        # تابع فوق هم dict و هم لیست را پشتیبانی می‌کند.
        return tavily_results_to_documents(resp if isinstance(resp, (dict, list)) else {"results": resp})

# ===== ساخت Retriever =====
retriever = TavilyRetriever(tavily)

# ===== تست با یک پرس و جو نمونه =====
sample_q = "Who is the current president of Bolivia?"
docs = retriever.get_relevant_documents(sample_q)

print("Docs retrieved:", len(docs))
for i, d in enumerate(docs[:5], 1):
    print(f"\nResult #{i}")
    print(" Title:", d.metadata.get("title"))
    print(" URL:", d.metadata.get("source_url"))
    # چاپ 300 کاراکتر اول از محتوا برای نمونه
    snippet = d.page_content.replace("\n", " ")[:300]
    print(" Snippet:", snippet + ("..." if len(d.page_content) > 300 else ""))





# ***chain_fallback***

In [ ]:
import os
os.environ["OPENROUTER_API_KEY"] = ""

from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

fallback_system = """You are an assistant with limited access to tools. You have been asked a user query and must
respond explaining that you cannot answer from the available knowledge/tools, or give a best-effort brief answer
within your allowed scope. Be concise, mention that this answer may be incomplete, and (if possible) suggest using
the SearchEngine for up-to-date facts.
"""
fallback_user = """Conversation history:
{history}

User question:
{user_query}

Return a concise answer (no more than 400 tokens)."""

fallback_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(fallback_system),
    HumanMessagePromptTemplate.from_template(fallback_user)
])

fallback_chain = LLMChain(
    llm=ChatOpenAI(
        model="meta-llama/llama-3.3-70b-instruct:free",
        temperature=0.0,
        max_tokens=512,
        openai_api_base="https://openrouter.ai/api/v1",
        openai_api_key=os.environ["OPENROUTER_API_KEY"]
    ),
    prompt=fallback_prompt
)

# Test fallback
print(fallback_chain.run({"history":"User: Hi\nAssistant: Hello", "user_query":"Who is the president of Bolivia?"})[:800])

# ***chain_context_with_generate***

In [ ]:
import os
os.environ["OPENROUTER_API_KEY"] = ""

from langchain.chat_models import ChatOpenAI

context_system = """You are an assistant that must answer a user question using only the provided documents.
Cite the sources inline by including [source_url] when relevant. If the documents are insufficient to answer, say so.
Be concise and helpful.
"""
context_user = """User question:
{user_query}

Relevant documents:
{documents}

Produce a clear answer and, if applicable, quote or paraphrase the relevant doc snippets and include their urls as [source_url].
"""

context_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(context_system),
    HumanMessagePromptTemplate.from_template(context_user)
])

context_chain = LLMChain(
    llm=ChatOpenAI(
        model="meta-llama/llama-3.3-70b-instruct:free",
        temperature=0.0,
        max_tokens=512,
        openai_api_base="https://openrouter.ai/api/v1",
        openai_api_key=os.environ["OPENROUTER_API_KEY"]
    ),
    prompt=context_prompt
)

# helper to run context chain
def run_context_with_docs(user_query: str, docs: list):
    # docs is a list of langchain.Document
    docs_text = "\n\n".join([f"---\nURL: {d.metadata.get('source_url', d.metadata.get('source_pdf','unknown'))}\nContent:\n{d.page_content[:1500]}" for d in docs])
    out = context_chain.run({"user_query": user_query, "documents": docs_text})
    return out

# Test: use retriever for a question in book domain
q = "what is transformer."
docs = retriever.get_relevant_documents(q)[:5]
print("Docs retrieved:", len(docs))
print(run_context_with_docs(q, docs)[:1000])


# ***Graphing and checking system performance***

In [ ]:
# نصب کتابخانه‌های لازم (در صورت نیاز)
!pip install -qU langgraph langchain langchain-community langchain-openai langchain-tavily faiss-cpu graphviz
!apt-get install -y graphviz > /dev/null

from io import BytesIO
from PIL import Image
from IPython.display import display, SVG
import os
import json
import re
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, END
from langchain.schema import Document

DEBUG = False

# --- State ---
class GraphState(TypedDict):
    query: str
    chat_history: List[str]
    documents: Optional[List[Document]]
    answer: Optional[str]
    route: Optional[str]

# --- Helper برای استخراج tool ---
ALLOWED_TOOLS = {"VectorStore", "SearchEngine", "None"}

def _find_allowed_in_string(s: str):
    if not isinstance(s, str):
        return None
    for a in ALLOWED_TOOLS:
        if a in s:
            return a
    return None

def extract_tool(decision) -> str:
    if decision is None:
        return "None"

    if isinstance(decision, (list, tuple)):
        for item in decision:
            t = extract_tool(item)
            if t in ALLOWED_TOOLS:
                return t
        return "None"

    if isinstance(decision, dict):
        if "tool" in decision:
            val = decision["tool"]
            if isinstance(val, str) and val in ALLOWED_TOOLS:
                return val
            return extract_tool(val)
        for v in decision.values():
            t = extract_tool(v)
            if t in ALLOWED_TOOLS:
                return t
        return "None"

    if isinstance(decision, str):
        s = decision.strip()
        s = re.sub(r'```(?:json)?', '', s).strip('`').strip()
        try:
            parsed = json.loads(s)
            return extract_tool(parsed)
        except Exception:
            pass
        found = _find_allowed_in_string(s)
        if found:
            return found
        if s in ALLOWED_TOOLS:
            return s
        return "None"

    if hasattr(decision, "tool"):
        try:
            return extract_tool(getattr(decision, "tool"))
        except Exception:
            pass

    if hasattr(decision, "dict"):
        try:
            return extract_tool(decision.dict())
        except Exception:
            pass

    return "None"

# --- نودها ---
def node_router(state: GraphState):
    payload = {"query": state["query"], "user_query": state["query"]}
    decision = None
    try:
        decision = router_chain.invoke(payload)
    except Exception:
        try:
            decision = router_chain.run(payload)
        except Exception:
            decision = None

    route = extract_tool(decision)
    state["route"] = route
    if DEBUG:
        print("=== router raw decision ===", decision)
        print("=== extracted route ===", route)
    return state

def store_vector(state: GraphState):
    docs = chain_retriever.invoke({"query": state["query"]})
    state["documents"] = docs
    return state

def engine_search(state: GraphState):
    docs = chain_engine_search.invoke({"query": state["query"]})
    state["documents"] = docs
    return state

def fallback(state: GraphState):
    answer = fallback_chain.invoke({
        "query": state["query"],
        "chat_history": state["chat_history"]
    })
    state["answer"] = answer
    return state

def context_with_generate(state: GraphState):
    answer = chain_context_with_generate.invoke({
        "query": state["query"],
        "documents": state["documents"]
    })
    state["answer"] = answer
    return state

# --- ساخت گراف (بدون docs_filter) ---
graph = StateGraph(GraphState)

graph.add_node("node_router", node_router)
graph.add_node("store_vector", store_vector)
graph.add_node("engine_search", engine_search)
graph.add_node("fallback", fallback)
graph.add_node("context_with_generate", context_with_generate)

graph.set_entry_point("node_router")

graph.add_conditional_edges(
    "node_router",
    lambda state: state["route"],
    {
        "VectorStore": "store_vector",
        "SearchEngine": "engine_search",
        "None": "fallback"
    },
)

# مستقیم وصل کردن
graph.add_edge("store_vector", "context_with_generate")
graph.add_edge("engine_search", "context_with_generate")

graph.add_edge("fallback", END)
graph.add_edge("context_with_generate", END)

app = graph.compile()

# --- نمایش گراف ---
print("=== Mermaid Text ===")
mermaid_text = app.get_graph().draw_mermaid()
print(mermaid_text)

# رندر محلی با graphviz
try:
    import graphviz
    nodes = {}
    edges = []
    for line in mermaid_text.splitlines():
        line = line.strip()
        if not line or line.startswith(('---','config:','classDef','graph')):
            continue
        line = line.replace('&nbsp;', ' ')
        m = re.match(r'^([A-Za-z0-9_]+)\s*[\(\[](.*?)[\)\]]', line)
        if m:
            nid, raw_label = m.groups()
            label = re.sub(r'<[^>]+>', '', raw_label).strip() or nid
            nodes[nid] = label
            continue
        if '->' in line:
            line_nosemi = line.rstrip(';').strip()
            pos = line_nosemi.rfind('->')
            left, right = line_nosemi[:pos], line_nosemi[pos+2:]
            src = left.strip().split()[0] if left.strip() else None
            dst = right.strip().split()[0] if right.strip() else None
            left_after_src = left.strip()[len(src):].strip() if src else ''
            lab = re.sub(r'[-\.\|]+', ' ', left_after_src)
            lab = ' '.join(lab.split()).strip()
            edge_label = lab if lab else None
            if src and dst:
                edges.append((src, dst, edge_label))
    dot = graphviz.Digraph(format='svg')
    for nid, label in nodes.items():
        dot.node(nid, label)
    for src, dst, elab in edges:
        if src not in nodes:
            dot.node(src, src)
        if dst not in nodes:
            dot.node(dst, dst)
        if elab:
            dot.edge(src, dst, label=elab)
        else:
            dot.edge(src, dst)
    svg_bytes = dot.pipe(format='svg')
    display(SVG(svg_bytes.decode('utf-8')))
except Exception as e:
    print("⚠️ خطا در graphviz:", e)
    print("Mermaid بالا را در https://mermaid.live کپی کن")
